# 02b · Preprocesamiento: etiquetado meteorológico con AEMET

## Motivación / Objetivos

Para evaluar el sistema bajo **circunstancias climáticas adversas**, cada fotograma necesita una etiqueta meteorológica. Las cámaras no la proporcionan, así que se obtiene de la estación AEMET más cercana.

Objetivos:
- Asignar a cada cámara su estación AEMET más cercana con datos.
- Asignar a cada fotograma la observación más cercana en el tiempo (ambas en UTC).
- Derivar la etiqueta `weather` (sunny, cloudy, rain, fog, windy, unknown) con `src/meteo.py`.

La iluminación (día / crepúsculo / noche) **no** es una clase meteorológica: la calcula `02a` por elevación solar (`illumination`).

**Objetivos del TFM relacionados:** OE-7 (comportamiento bajo condiciones climáticas).

**Requisito:** API key gratuita de AEMET OpenData (https://opendata.aemet.es/centrodedescargas/altaUsuario) en el `.env` de la raíz del proyecto (`AEMET_API_KEY=...`, ver `.env.example`) o como variable de entorno.

## Método

Se cruzan las coordenadas de cada cámara (catálogo `da_camarasTrafico-4326.csv`) y la marca de tiempo UTC de cada fotograma con las observaciones de AEMET.

1. Configuración.
2. Fotogramas y coordenadas de las cámaras.
3. Inventario de estaciones AEMET y estación más cercana.
4. Descarga de observaciones meteorológicas.
5. Asignación de la observación a cada fotograma.
6. Derivación de la etiqueta `weather`.
7. Exportación de la fase `weather`.

### 1. Configuración

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from catalogo import load_catalog
from config import get_secret, load_config, path
from meteo import classify_weather, dms_to_decimal, haversine_km
from pipeline import load_phase, save_phase

CFG = load_config()["meteo"]
CACHE_DIR = path("meteo_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Variable de entorno o fichero .env de la raíz del proyecto (excluido de git, ver .env.example)
AEMET_API_KEY = get_secret("AEMET_API_KEY")
MAX_TIME_DIFF_MINUTES = CFG["max_diff_minutos"]
MAX_STATION_DISTANCE_KM = CFG["max_distancia_km"]

print(f"API key configurada: {'si' if AEMET_API_KEY else 'no (solo funcionara con datos en cache)'}")

API key configurada: si


### 2. Fotogramas y coordenadas de las cámaras

In [2]:
frames_df = load_phase("frames")
camaras_df = load_catalog()

print(f"Fotogramas: {len(frames_df)} | Cámaras en catálogo: {len(camaras_df)}")
print(f"Cámaras en frames: {frames_df['camera'].nunique()}")
display(camaras_df[["ID", "NOMBRE", "lat", "lon"]].head())

Fotogramas: 74508 | Cámaras en catálogo: 134
Cámaras en frames: 133


,ID,NOMBRE,lat,lon
0,21611,TV87-EL BONI - CAMPANILLAS,36.724063,-4.539063
1,75066,TV100-JOSÉ CALDERÓN - JUNTA DE DISTRITO CAMPAN...,36.728064,-4.544862
2,75086,TV120-HERMANOS LUMIERE - LICURGO,36.693708,-4.462710
3,75093,TV127-CARRIL DEL CAPITÁN - JUAN DE ROBLES,36.721554,-4.461611
4,75069,TV103-AVDA LOS GUINDOS - CMNO DE LA TÉRMICA,36.690822,-4.447017


### 3. Inventario de estaciones AEMET y estación más cercana

El inventario da `latitud`/`longitud` en grados-minutos-segundos con hemisferio (`364000N`, `042900W`). `pd.to_numeric` las convertía en NaN y por eso no se asignaba ninguna estación; ahora se convierten con `meteo.dms_to_decimal`. La distancia es la de Haversine.

In [3]:
AEMET_BASE = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key": AEMET_API_KEY, "Accept": "application/json"}


def aemet_get(endpoint, params=None, cache_name=None):
    """Llama a un endpoint AEMET (dos pasos: obtener URL de datos y descargar datos) con cache en disco."""
    if cache_name:
        cache_path = CACHE_DIR / cache_name
        if cache_path.exists():
            return json.loads(cache_path.read_text(encoding="utf-8"))

    if not AEMET_API_KEY:
        raise RuntimeError("Falta AEMET_API_KEY: añádela al fichero .env de la raíz del proyecto (ver .env.example).")

    r = requests.get(f"{AEMET_BASE}{endpoint}", headers=HEADERS, params=params or {}, timeout=60)
    r.raise_for_status()
    meta = r.json()
    if meta.get("estado") != 200 or "datos" not in meta:
        raise RuntimeError(f"Respuesta AEMET inesperada: {meta}")

    r2 = requests.get(meta["datos"], timeout=120)
    r2.raise_for_status()
    data = r2.json()
    if cache_name:
        cache_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    return data


estaciones = aemet_get(
    "/valores/climatologicos/inventarioestaciones/todasestaciones",
    cache_name="inventario_estaciones.json",
)
estaciones_df = pd.DataFrame(estaciones)
estaciones_df["latitud"] = estaciones_df["latitud"].map(dms_to_decimal)
estaciones_df["longitud"] = estaciones_df["longitud"].map(dms_to_decimal)
estaciones_df = estaciones_df.dropna(subset=["latitud", "longitud"])

print(f"Estaciones en inventario: {len(estaciones_df)}")
display(estaciones_df[["indicativo", "nombre", "provincia", "latitud", "longitud"]].head())

Estaciones en inventario: 924


,indicativo,nombre,provincia,latitud,longitud
0,B013X,"ESCORCA, LLUC",ILLES BALEARS,39.823333,2.885833
1,B051A,"SÓLLER, PUERTO",BALEARES,39.795556,2.691389
2,B087X,BANYALBUFAR,BALEARES,39.689167,2.512778
3,B103B,ANDRATX - SANT ELM,BALEARES,39.579444,2.368889
4,B158X,"CALVIÀ, ES CAPDELLÀ",BALEARES,39.551389,2.466389


In [4]:
def nearest_station(lat, lon, stations):
    distances = [haversine_km(lat, lon, s_lat, s_lon) for s_lat, s_lon in zip(stations["latitud"], stations["longitud"])]
    best = int(np.argmin(distances))
    return stations.iloc[best], distances[best]


def assign_stations(cams, stations):
    """Estación más cercana a cada cámara (sin estación si está a más de MAX_STATION_DISTANCE_KM)."""
    nearest = [nearest_station(lat, lon, stations) for lat, lon in zip(cams["lat"], cams["lon"])]
    assigned = pd.DataFrame({
        "camera": cams["ID"].values,
        "estacion_id": [s["indicativo"] for s, _ in nearest],
        "estacion_nombre": [s["nombre"] for s, _ in nearest],
        "distancia_estacion_km": [d for _, d in nearest],
    })
    assigned.loc[assigned["distancia_estacion_km"] > MAX_STATION_DISTANCE_KM, "estacion_id"] = None
    return assigned


# Asignación preliminar con todo el inventario; se rehace en la sección 5 solo con las
# estaciones que tienen observaciones en el periodo
cam_coords = camaras_df.dropna(subset=["lat", "lon"])
camera_station = assign_stations(cam_coords, estaciones_df)

print(f"Cámaras con estación asignada: {camera_station['estacion_id'].notna().sum()} de {len(camera_station)}")
print(f"Estaciones distintas usadas: {camera_station['estacion_id'].nunique()}")
print(f"Distancia media: {camera_station['distancia_estacion_km'].mean():.1f} km | máxima: {camera_station['distancia_estacion_km'].max():.1f} km")
display(camera_station.sort_values("distancia_estacion_km", ascending=False).head(10))

Cámaras con estación asignada: 134 de 134
Estaciones distintas usadas: 3
Distancia media: 2.2 km | máxima: 6.3 km


,camera,estacion_id,estacion_nombre,distancia_estacion_km
111,21606,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",6.335728
131,21541,6172O,"MÁLAGA, PUERTO",5.959511
1,75066,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",5.747388
128,75100,6172O,"MÁLAGA, PUERTO",5.522646
13,21542,6172O,"MÁLAGA, PUERTO",5.199537
0,21611,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",5.163202
8,75099,6172O,"MÁLAGA, PUERTO",4.808219
89,78680,6172O,"MÁLAGA, PUERTO",4.776185
10,21574,6172O,"MÁLAGA, PUERTO",4.064860
9,21543,6172O,"MÁLAGA, PUERTO",4.050923


### 4. Descarga de observaciones meteorológicas

Dos modos, elegidos automáticamente según la antigüedad de los fotogramas:
- **Últimas observaciones** (`/observacion/convencional/todas`): si los fotogramas son de las últimas ~24 h.
- **Climatológicas diarias** por rango de fechas: para fotogramas históricos (resolución diaria). Se consultan en tramos de 14 días, porque AEMET limita cada consulta a 15.

In [5]:
ts_min, ts_max = frames_df["timestamp"].min(), frames_df["timestamp"].max()
now_utc = pd.Timestamp.now(tz="UTC").tz_localize(None)
horas_desde_min = (now_utc - ts_min).total_seconds() / 3600 if pd.notna(ts_min) else np.inf

use_ultimas = horas_desde_min <= 24
print(f"Rango timestamps (UTC): {ts_min} -> {ts_max}")
print(f"Modo descarga: {'ultimas observaciones' if use_ultimas else 'climatologicas diarias (rango)'}")

obs_records = []
if use_ultimas:
    obs = aemet_get("/observacion/convencional/todas", cache_name=f"obs_ultimas_{now_utc:%Y%m%dT%H}.json")
    for o in obs:
        obs_records.append({
            "estacion_id": o.get("idema"),
            "fint": pd.to_datetime(o.get("fint"), errors="coerce", utc=True).tz_localize(None) if o.get("fint") else pd.NaT,
            "prec": o.get("prec"),   # precipitacion acumulada (mm) en la ultima hora
            "vv": o.get("vv"),       # velocidad media del viento (m/s)
            "hr": o.get("hr"),       # humedad relativa (%)
            "ta": o.get("ta"),       # temperatura (C)
            "vis": o.get("vis"),     # visibilidad (m), si esta disponible
            "nub": o.get("nub"),     # nubosidad (octas), si esta disponible
        })
else:
    # AEMET limita cada consulta de datos diarios a 15 días: se pide en tramos de 14 días
    obs = []
    tramo_ini = ts_min.normalize()
    while tramo_ini <= ts_max:
        tramo_fin = min(tramo_ini + pd.Timedelta(days=14) - pd.Timedelta(seconds=1), ts_max.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59))
        fecha_ini = tramo_ini.strftime("%Y-%m-%dT%H:%M:%SUTC")
        fecha_fin = tramo_fin.strftime("%Y-%m-%dT%H:%M:%SUTC")
        obs += aemet_get(
            f"/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/todasestaciones",
            cache_name=f"obs_diarias_{tramo_ini.date()}_{tramo_fin.date()}.json",
        )
        print(f"  Tramo {tramo_ini.date()} -> {tramo_fin.date()}: {len(obs)} observaciones acumuladas")
        tramo_ini += pd.Timedelta(days=14)
    for o in obs:
        obs_records.append({
            "estacion_id": o.get("indicativo"),
            # Dato diario: se sitúa a mediodía UTC para el cruce temporal
            "fint": pd.to_datetime(o.get("fecha"), errors="coerce") + pd.Timedelta(hours=12),
            "prec": o.get("prec"),
            "vv": o.get("velmedia"),
            "hr": o.get("hrMedia"),
            "ta": o.get("tmed"),
            "vis": None,
            "nub": None,
        })

obs_df = pd.DataFrame(obs_records)
for col in ["prec", "vv", "hr", "ta", "vis", "nub"]:
    obs_df[col] = pd.to_numeric(obs_df[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
obs_df = obs_df.dropna(subset=["estacion_id", "fint"])

# En datos diarios la ventana temporal debe cubrir el día completo
max_diff = MAX_TIME_DIFF_MINUTES if use_ultimas else 12 * 60
print(f"Observaciones descargadas: {len(obs_df)} | ventana de cruce: {max_diff} min")
display(obs_df.head())

Rango timestamps (UTC): 2026-08-25 14:48:08 -> 2026-10-17 13:40:57
Modo descarga: climatologicas diarias (rango)


  Tramo 2026-08-25 -> 2026-09-07: 11701 observaciones acumuladas


  Tramo 2026-09-08 -> 2026-09-21: 23346 observaciones acumuladas


  Tramo 2026-09-22 -> 2026-10-05: 34966 observaciones acumuladas


  Tramo 2026-10-06 -> 2026-10-17: 42390 observaciones acumuladas


Observaciones descargadas: 42390 | ventana de cruce: 720 min


,estacion_id,fint,prec,vv,hr,ta,vis,nub
0,8293X,2026-08-25 12:00:00,0.0,2.5,40.0,29.5,NaN,NaN
1,5612B,2026-08-25 12:00:00,0.0,1.7,52.0,22.4,NaN,NaN
2,2462,2026-08-25 12:00:00,0.0,3.1,64.0,13.6,NaN,NaN
3,5612X,2026-08-25 12:00:00,0.0,NaN,56.0,22.9,NaN,NaN
4,3094B,2026-08-25 12:00:00,0.0,2.8,44.0,23.2,NaN,NaN


### 5. Asignación de la observación a cada fotograma

La estación se elige solo entre las que tienen observaciones en el periodo. Algunas cercanas (p. ej. MÁLAGA, PUERTO) no publican datos diarios y dejaban la mayoría de los fotogramas como `unknown`. Se descartan las estaciones a más de `meteo.max_distancia_km`.

In [6]:
# Solo se usan estaciones con observaciones en el periodo: algunas cercanas (p. ej. MÁLAGA, PUERTO)
# no publican datos climatológicos diarios y dejarían todos sus fotogramas como unknown
estaciones_con_datos = estaciones_df[estaciones_df["indicativo"].isin(obs_df["estacion_id"].unique())]
camera_station = assign_stations(cam_coords, estaciones_con_datos)
print(f"Estaciones con datos: {len(estaciones_con_datos)} | usadas: {camera_station['estacion_id'].nunique()}")
display(camera_station.groupby(["estacion_id", "estacion_nombre"]).size().rename("camaras").to_frame())

frames_w = frames_df[["camera", "image", "timestamp"]].merge(camera_station, on="camera", how="left")
obs_by_station = {k: g.sort_values("fint") for k, g in obs_df.groupby("estacion_id")}


def nearest_observation(station_id, ts):
    if pd.isna(ts) or station_id not in obs_by_station:
        return None, np.nan
    g = obs_by_station[station_id]
    diffs = (g["fint"] - ts).abs()
    i = diffs.idxmin()
    diff_min = diffs.loc[i].total_seconds() / 60.0
    if diff_min > max_diff:
        return None, diff_min
    return g.loc[i], diff_min


assigned = [nearest_observation(s, ts) for s, ts in zip(frames_w["estacion_id"], frames_w["timestamp"])]
frames_w["obs_asignada"] = [a[0] is not None for a in assigned]
frames_w["diff_obs_minutos"] = [a[1] for a in assigned]
for col in ["prec", "vv", "hr", "ta", "vis", "nub"]:
    frames_w[col] = [a[0][col] if a[0] is not None else np.nan for a in assigned]

n_ok = int(frames_w["obs_asignada"].sum())
print(f"Fotogramas con observación asignada: {n_ok} de {len(frames_w)} ({100 * n_ok / len(frames_w):.1f}%)")
display(frames_w.head(10))

Estaciones con datos: 843 | usadas: 3


,,camaras
estacion_id,estacion_nombre,
6155A,MÁLAGA AEROPUERTO,9
6156X,"MÁLAGA, CENTRO METEOROLÓGICO",118
6175X,RINCÓN DE LA VICTORIA,7


Fotogramas con observación asignada: 57433 de 74508 (77.1%)


,camera,image,timestamp,estacion_id,estacion_nombre,distancia_estacion_km,obs_asignada,diff_obs_minutos,prec,vv,hr,ta,vis,nub
0,21536,20261012T230457Z.jpg,2026-10-12 23:04:57,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",6.185246,True,664.950000,0.0,2.2,75.0,21.6,NaN,NaN
1,21538,20261012T230623Z.jpg,2026-10-12 23:06:23,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",2.954556,True,666.383333,0.0,2.2,75.0,21.6,NaN,NaN
2,21539,20261012T230624Z.jpg,2026-10-12 23:06:24,6156X,"MÁLAGA, CENTRO METEOROLÓGICO",3.590879,True,666.400000,0.0,2.2,75.0,21.6,NaN,NaN
3,21541,20260825T145146Z.jpg,2026-08-25 14:51:46,6175X,RINCÓN DE LA VICTORIA,5.988965,True,171.766667,0.0,NaN,54.0,24.8,NaN,NaN
4,21541,20260825T155429Z.jpg,2026-08-25 15:54:29,6175X,RINCÓN DE LA VICTORIA,5.988965,True,234.483333,0.0,NaN,54.0,24.8,NaN,NaN
5,21541,20260825T155459Z.jpg,2026-08-25 15:54:59,6175X,RINCÓN DE LA VICTORIA,5.988965,True,234.983333,0.0,NaN,54.0,24.8,NaN,NaN
6,21541,20260825T155529Z.jpg,2026-08-25 15:55:29,6175X,RINCÓN DE LA VICTORIA,5.988965,True,235.483333,0.0,NaN,54.0,24.8,NaN,NaN
7,21541,20260825T155559Z.jpg,2026-08-25 15:55:59,6175X,RINCÓN DE LA VICTORIA,5.988965,True,235.983333,0.0,NaN,54.0,24.8,NaN,NaN
8,21541,20260825T155629Z.jpg,2026-08-25 15:56:29,6175X,RINCÓN DE LA VICTORIA,5.988965,True,236.483333,0.0,NaN,54.0,24.8,NaN,NaN
9,21541,20260825T155659Z.jpg,2026-08-25 15:56:59,6175X,RINCÓN DE LA VICTORIA,5.988965,True,236.983333,0.0,NaN,54.0,24.8,NaN,NaN


### 6. Derivación de la etiqueta `weather`

Reglas (`meteo.classify_weather`, por orden de prioridad):
1. `rain`: precipitación > 0 mm.
2. `fog`: visibilidad < 1000 m o humedad relativa ≥ 97 %.
3. `windy`: viento medio ≥ 10 m/s.
4. `cloudy`: nubosidad ≥ 6 octas o humedad relativa ≥ 85 %.
5. `sunny`: resto con observación válida.
6. `unknown`: sin observación asignable.

In [7]:
frames_w["weather"] = [
    classify_weather(prec=r.prec, vis=r.vis, hr=r.hr, wind=r.vv, nub=r.nub) if r.obs_asignada else "unknown"
    for r in frames_w.itertuples()
]
print("Distribución weather:")
print(frames_w["weather"].value_counts(dropna=False))

Distribución weather:
weather
sunny      57433
unknown    17075
Name: count, dtype: int64


### 7. Exportación de la fase `weather`

In [8]:
export_cols = [
    "camera", "image", "weather",
    "estacion_id", "estacion_nombre", "distancia_estacion_km", "diff_obs_minutos",
    "prec", "vv", "hr", "ta", "vis", "nub",
]
out = save_phase(frames_w[export_cols], "weather")
print(f"Fase weather: {len(frames_w)} fotogramas -> {out}")
print(f"Fotogramas con weather desconocido: {(frames_w['weather'] == 'unknown').sum()}")

Fase weather: 74508 fotogramas -> C:\Users\loren\OneDrive\Escritorio\MASTER-UIMP\TFM\tfm\data\malaga\fases\weather.parquet
Fotogramas con weather desconocido: 17075


## Salida / Resultados

| Salida | Contenido |
|---|---|
| `data/malaga/fases/weather.parquet` | Etiqueta `weather`, estación asignada y observación por fotograma. |

**Última ejecución (fotogramas del 25/08 al 17/10/2026):**
- Modo de descarga: climatológicas diarias, 42.390 observaciones en 4 tramos.
- 843 estaciones con datos, 3 usadas: Centro Meteorológico (118 cámaras), Aeropuerto (9) y Rincón de la Victoria (7).
- 57.433 fotogramas (77,1 %) con observación, **todos `sunny`**. 17.075 (22,9 %) quedan como `unknown`.

## Observaciones

- Con datos diarios, la etiqueta describe el día, no el momento exacto de la imagen.
- Los `unknown` corresponden a días que AEMET aún no había publicado (los últimos de la captura). Basta con volver a ejecutar el notebook unos días después.
- **Limitación para OE-7:** en todo el periodo no hay lluvia, niebla ni viento. Las condiciones adversas disponibles son la noche y el crepúsculo (`illumination`). Si llueve durante la captura, conviene crear un lote específico.
- `notebooks/README.md` todavía cita un 98,4 % con clima asignado, de una ejecución anterior con menos datos.